# Lab 2\. Inspecting and Modifying a PyTorch Model

## Overview

In Chapter 2 you looked inside a PyTorch model: you saw that `nn.Module` is the container that holds and tracks a model's layers, that a Linear layer transforms a vector of features into a different vector of features, and that an activation function such as ReLU adds the non-linearity a network needs to learn complex patterns. This lab is where you stop reading about those parts and start handling them yourself.

Working in a notebook, you will build the `DigitClassifier` model from the chapter, open it up with PyTorch's own inspection tools, and watch data change shape as it moves through the model. Then you will change the model in two small, deliberate ways: you will add a layer, and you will swap one activation function for another. After each change you will inspect the model again and read what changed. The dataset behind the model is MNIST, a collection of 28-by-28 grayscale images of handwritten digits from 0 to 9\.

This is the everyday loop of model development. When you join a team working in PyTorch, you will spend far more time reading, adjusting, and re-inspecting existing models than writing new ones from a blank file. Being able to open a model, confirm it is built the way you expect, make one change, and see the effect is the habit this lab builds. Later chapters put this model into training; here you make sure you can see and shape it first.

## Learning Objectives

By the end of this lab, you should be able to:

* Define a simple neural network by subclassing `nn.Module`, placing layers in `__init__` and the data flow in `forward`.  
* Inspect a model's architecture with `print(model)` and its learnable values with `model.parameters()` and `model.named_parameters()`.  
* Trace how a tensor's shape changes as it passes through a Linear layer, and pass an MNIST image through the full model.  
* Compare the output of ReLU and Leaky ReLU on the same input tensor.  
* Modify a model by adding a layer and by swapping its activation function, and read the effect of each change on the model's structure and parameter count.

## Prerequisites

This course’s labs are **not cumulative**, so this lab does not depend on any prior resources or the state of earlier labs if there are any.

You need the following before you start:

- **A notebook environment:** either a Google account for Google Colab, or a local installation of Jupyter Notebook. Every step in this lab runs on a CPU; no GPU is required.  
- **Python 3.12 or later.** On Colab this is already provided. For a local environment, confirm your version with `python --version`.  
- **PyTorch 2.12.1 and TorchVision 0.17 or later (0.27 preferred),** installed with `pip` in the first setup step below. If you are on Colab, these may already be present; the setup step confirms the versions. The `Imagenette` dataset requires torchvision 0.17 or newer.  
- **A web browser** (Chrome, Firefox, or Edge are recommended for Colab).

## Estimated Time

35–45 minutes.

## How to Perform the Exercise

You will work entirely in one notebook, running cells from top to bottom. Create a new notebook in Google Colab (**File \> New notebook**) or a new `.ipynb` in local Jupyter. Download the chapter 2 lab notebook from the [LFD2002-resources GitHub repository](https://github.com/lftraining/LFD2002-resources.git) from the **LFD2002-resources/labs** directory. In Google Colab, select File, then Upload Notebook, and select the chapter 2 lab notebook you downloaded.

Read the explanation before running each cell, run it, and compare what you see to the Sample Output or Output shown after each block. Sample Output means the exact values will differ from run to run (random tensors, dataset counts, file paths), so match the shape and structure rather than every character. Output marks a checkpoint where a specific outcome confirms the step worked.

Commands in this lab use the notebook form, where a line beginning with `!` runs a shell command from inside a notebook cell (for example, `!pip install ...`). If you run these in a local terminal instead of a notebook cell, drop the leading `!`.

> **Note:** Run the cells in order and only once each, unless a step says otherwise. If you restart the notebook's runtime, re-run the cells from the top, because the variables live in memory and are cleared on restart.

## Setup: Environment Preparation

The step in this section only prepares the environment. It is not the learning content of the lab, and on a managed course platform it may already be done for you. Run it once at the top of your notebook, then move on to Part A.

### Setup 1\. Install torch and torchvision

Install the two libraries the lab needs. TorchVision is the PyTorch companion library for computer vision, and it provides the MNIST dataset you will load. On Colab a recent version is usually already present, in which case pip reports that the requirement is already satisfied and does nothing further.


In [ ]:
!pip install torch torchvision


### Setup 2\. Import PyTorch and Load a Sample from MNIST

You need PyTorch's core package and its neural-network module, plus Torchvision to fetch MNIST. Create the cell with the imports and a one-time download of the dataset. The`ToImage() and ToDtype()` steps convert each image from its original picture format into a PyTorch tensor, which is the form the model can accept.


In [ ]:
import torch
import torch.nn as nn
from torchvision import datasets
from torchvision.transforms import v2

transform = v2.Compose([
    v2.ToImage(),                           # PIL image -> image tensor
    v2.ToDtype(torch.float32, scale=True),  # cast to float32, scale pixels to [0.0, 1.0]
])

# Download MNIST once and convert each image to a tensor.
train_data = datasets.MNIST(root="data", train=True, download=True, transform=transform)

# Take a single image and its label so we have real data to work with later.
image, label = train_data[0]
print("Image tensor shape:", image.shape)
print("Label:", label)


The first time you run this cell, Torchvision downloads MNIST into a local `data` directory, which takes a few moments. After that it loads instantly. The two print lines confirm the download worked and show the shape of one image.

**Output:**

```
Image tensor shape: torch.Size([1, 28, 28])
Label: 5
```

The shape `[1, 28, 28]` means one color channel (grayscale), 28 rows, and 28 columns. Keep this shape in mind; it explains why the model starts by flattening.

To print the single image, run:


In [ ]:
import matplotlib.pyplot as plt
from torchvision.transforms import v2

# Convert tensor back to a PIL Image
to_pil = v2.ToPILImage()
pil_image = to_pil(image)

# Display it in Colab
plt.imshow(pil_image, cmap="gray")
plt.title(f"Label: {label}")
plt.axis("off")
plt.show()


## Part A: Define and Inspect the Model

### Step 1\. Define the DigitClassifier

Before you can inspect a model, you need one to inspect. Recreate the exact `DigitClassifier` model from the chapter. It subclasses `nn.Module`, sets up its layers in `__init__`, and describes the data flow in `forward`. The `nn.Flatten()` layer turns each `[1, 28, 28]` image into a flat vector of 784 values so the first Linear layer can accept it, since 28 times 28 is 784\.


In [ ]:
class DigitClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.layer1 = nn.Linear(784, 128)
        self.activation = nn.ReLU()
        self.layer2 = nn.Linear(128, 10)

    def forward(self, x):
        x = self.flatten(x)
        x = self.layer1(x)
        x = self.activation(x)
        x = self.layer2(x)
        return x

model = DigitClassifier()


Running this cell defines the class and builds one model object named `model`. Nothing prints, because creating the model only runs the setup in `__init__`; no data has moved through it yet. The final Linear layer produces 10 outputs, one score per digit class from 0 to 9\.

### Step 2\. Inspect the Architecture with print(model)

Because `nn.Module` registers every layer you assign in `__init__`, you can ask the model to describe itself. Pass it to `print` to list its registered layers and how each is configured.


In [ ]:
print(model)


**Output:**

```
DigitClassifier(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (layer1): Linear(in_features=784, out_features=128, bias=True)
  (activation): ReLU()
  (layer2): Linear(in_features=128, out_features=10, bias=True)
)
```

Each line names a layer by the attribute name you gave it and shows its settings. The `layer1` entry confirms it accepts 784 input features and produces 128, and `layer2` confirms it takes those 128 and produces 10\. This printout is the fastest way to confirm a model was built the way you intended.

### Step 3\. Inspect the Learnable Parameters

The printed structure shows the layers, but not the numbers the model will learn. Those live in the model's parameters: the weights and biases of each Linear layer. The `model.named_parameters()` method returns each parameter together with its name, and `.numel()` reports how many individual values a parameter holds. Use them to list the shape of every parameter and to total them up.


In [ ]:
total = 0
for name, param in model.named_parameters():
    count = param.numel()
    total += count
    print(f"{name:16} shape {tuple(param.shape)}  ->  {count} values")

print(f"\nTotal learnable parameters: {total}")


**Output:**

```
layer1.weight    shape (128, 784)  ->  100352 values
layer1.bias      shape (128,)  ->  128 values
layer2.weight    shape (10, 128)  ->  1280 values
layer2.bias      shape (10,)  ->  10 values

Total learnable parameters: 101770
```

Two things are worth reading here. Only the Linear layers appear, because `flatten` and the ReLU activation have no learnable values of their own. And each weight matrix has the shape `(out_features, in_features)` you would expect from the chapter: `layer1.weight` is `(128, 784)` because it maps 784 inputs to 128 outputs. The total, 101,770, is the full set of numbers that training will later adjust.

## Part B: See What Layers and Activations Do to Data

### Step 4\. Trace a Tensor's Shape Through a Linear Layer

A Linear layer's job is to change the number of features while leaving the batch of samples alone. The clearest way to see this is to build one small Linear layer on its own and watch a tensor's shape before and after. This example mirrors the one from the chapter: a layer that turns 20 features into 30, given a batch of 128 samples.


In [ ]:
small_layer = nn.Linear(20, 30)
sample = torch.randn(128, 20)   # 128 samples, each with 20 features
result = small_layer(sample)

print("Input shape: ", sample.shape)
print("Output shape:", result.shape)


**Output:**

```
Input shape:  torch.Size([128, 20])
Output shape: torch.Size([128, 30])
```

The batch of 128 is untouched, while the feature dimension changed from 20 to 30 because that is what the layer was configured to do. This is exactly how `layer1` in your model turns each image's 784 features into 128\.

#### Step 4.1. Comparison: How a Conv2d Layer Treats the Same Data

This short comparison is for contrast only; it is not part of the model you are building here. The chapter explained that a convolutional layer keeps an image's spatial layout instead of flattening it. You can see that difference in the output shape. Pass a single-channel 28-by-28 image, with a batch dimension added, through a `Conv2d` layer.


In [ ]:
conv = nn.Conv2d(in_channels=1, out_channels=16, kernel_size=3)
image_batch = image.unsqueeze(0)          # add a batch dimension: [1, 1, 28, 28]
conv_out = conv(image_batch)

print("Conv input shape: ", image_batch.shape)
print("Conv output shape:", conv_out.shape)


**Output:**

```
Conv input shape:  torch.Size([1, 1, 28, 28])
Conv output shape: torch.Size([1, 16, 26, 26])
```

The output still has a height and a width, unlike the flat vector a Linear layer would give. The channel count went from 1 to 16 because you asked for 16 feature maps, and the height and width shrank slightly because a 3-by-3 window cannot start at the very edge. The point is only the contrast: `Conv2d` preserves spatial structure, while a Linear layer requires the flattened vector your `DigitClassifier` produces.

### Step 5\. Pass an MNIST Image Through the Model

Now send a real image through the full `DigitClassifier`. The model expects a batch, so add a batch dimension to the single image with `unsqueeze(0)`, turning its shape from `[1, 28, 28]` into `[1, 1, 28, 28]`. The model flattens it, passes it through both Linear layers and the activation, and returns one score per digit class.


In [ ]:
single = image.unsqueeze(0)     # shape [1, 1, 28, 28]
output = model(single)

print("Model output shape:", output.shape)
print("Raw scores:", output)


**Output (shape):**

```
Model output shape: torch.Size([1, 10])
```

**Sample Output (values):**

```
Raw scores: tensor([[ 0.03, -0.11,  0.07,  0.02, -0.05,  0.09, -0.02,  0.14, -0.08,  0.01]],
       grad_fn=<AddmmBackward0>)
```

The output shape `[1, 10]` is the important part: one row for the single image, and ten values in that row, one score for each digit from 0 to 9\. The actual numbers are meaningless right now because the model's weights are still random; it has not been trained. Training those weights so the scores become useful is the subject of Chapter 3\. Notice that you called `model(single)`, not `model.forward(single)`, which is the pattern the chapter emphasized.

### Step 6\. Compare ReLU and Leaky ReLU on the Same Tensor

Activation functions are easiest to understand by feeding one tensor through each and comparing the results. Create a tensor that mixes positive and negative values, then apply ReLU and Leaky ReLU to it. This is the same input the chapter used, so you can check your results against it.


In [ ]:
values = torch.tensor([8., -3., 0., 1., 5., -2., -1., 4.])

relu = nn.ReLU()
leaky = nn.LeakyReLU()   # negative_slope defaults to 0.01

print("Input:     ", values)
print("ReLU:      ", relu(values))
print("Leaky ReLU:", leaky(values))


**Output:**

```
Input:      tensor([ 8., -3.,  0.,  1.,  5., -2., -1.,  4.])
ReLU:       tensor([8., 0., 0., 1., 5., 0., 0., 4.])
Leaky ReLU: tensor([ 8.0000, -0.0300,  0.0000,  1.0000,  5.0000, -0.0200, -0.0100,  4.0000])
```

Compare the two output rows against the input. ReLU left every positive value alone and replaced every negative value with exactly zero. Leaky ReLU also kept the positives, but instead of zeroing the negatives it multiplied them by its default negative slope of `0.01`, so `-3` became `-0.03` and `-2` became `-0.02`. That small non-zero response is the entire difference between the two, and it is why Leaky ReLU can help when ReLU units get stuck outputting zero.

## Part C: Modify the Model

You will now make two independent changes to the original `DigitClassifier`, each in its own cell. Keeping them separate lets you see the effect of one change at a time.

### Step 7\. Add a Hidden Layer

The chapter's key point about Linear layers is that each layer's `in_features` must match the number of features arriving from the layer before it. Adding a layer is where that rule becomes real: the new layer has to fit the chain. Define a deeper version of the classifier that inserts one more Linear layer (self.layer2), turning the path from 784 to 128 to 64 to 10 features, with an activation after each hidden layer (add self.activation2). Write the code for these two additions yourself, in the provided model class.


In [ ]:
class DeeperClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.layer1 = nn.Linear(784, 128)
        self.activation1 = nn.ReLU()
    	 # Add a new hidden layer - replace ... with a Linear layer
        self.layer2 = ... # write your code here
    	 # Add a new activation function - replace ... with a new function
        self.activation2 = ... # write your code here
        self.layer3 = nn.Linear(64, 10)

    def forward(self, x):
        x = self.flatten(x)
        x = self.layer1(x)
        x = self.activation1(x)
        x = self.layer2(x)
        x = self.activation2(x)
        x = self.layer3(x)
        return x

deeper_model = DeeperClassifier()
print(deeper_model)
output = deeper_model(single)
print(output)


**Output (model):**

```
DeeperClassifier(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (layer1): Linear(in_features=784, out_features=128, bias=True)
  (activation1): ReLU()
  (layer2): Linear(in_features=128, out_features=64, bias=True)
  (activation2): ReLU()
  (layer3): Linear(in_features=64, out_features=10, bias=True)
)
```

**Sample Output (correct):**

```
tensor([[-0.0769, -0.0783, -0.0617, -0.0252, -0.0878,  0.1548, -0.1016,  0.0964,
          0.0957, -0.0544]], grad_fn=<AddmmBackward0>)
```

**Sample Output (error):**

```
RuntimeError: mat1 and mat2 shapes cannot be multiplied (1x128 and 100x64)
```

If your model is correct, the second print statement will produce an output containing 10 logits. However, if the model was incorrectly built (e.g. taking 100 input features instead), you’ll see an error message because of the mismatched shapes.

Look at how the feature counts line up: `layer1` outputs 128, so the new `layer2` must accept 128, and it outputs 64, so `layer3` must accept 64\. If you had set `layer2` to accept anything other than 128, running an image through the model would raise a shape error. The final layer still outputs 10, because the task is still to score 10 digit classes.

You can confirm the model grew by counting its parameters again with the same loop from Step 3\.


In [ ]:
total = sum(p.numel() for p in deeper_model.parameters())
print(f"Total learnable parameters: {total}")


**Output:**

```
Total learnable parameters: 109386
```

The count rose from 101,770 to 109,386 because the extra layer added its own weights and biases. Adding capacity to a model means adding parameters, and this is how you check by how much.

### Step 8\. Swap the Activation Function

The second change is smaller: replace the activation function without touching any layer. Define a version of the original two-layer classifier that uses Leaky ReLU in place of ReLU. Only the activation line changes.


In [ ]:
class LeakyClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.layer1 = nn.Linear(784, 128)
	 # Replace the original activation function with a leaky one
        self.activation = ...
        self.layer2 = nn.Linear(128, 10)

    def forward(self, x):
        x = self.flatten(x)
        x = self.layer1(x)
        x = self.activation(x)
        x = self.layer2(x)
        return x

leaky_model = LeakyClassifier()
print(leaky_model)


**Output:**

```
LeakyClassifier(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (layer1): Linear(in_features=784, out_features=128, bias=True)
  (activation): LeakyReLU(negative_slope=0.01)
  (layer2): Linear(in_features=128, out_features=10, bias=True)
)
```

If you used the correct function, the `activation` line now reads `LeakyReLU(negative_slope=0.01)` instead of `ReLU()`, and the layers are unchanged. Count the parameters one more time to confirm something the chapter implied but did not measure.


In [ ]:
total = sum(p.numel() for p in leaky_model.parameters())
print(f"Total learnable parameters: {total}")


**Output:**

```
Total learnable parameters: 101770
```

The total is 101,770, the same as the original model. Swapping the activation changed how the model treats negative values, but it did not change the parameter count at all, because activation functions have no learnable parameters. That is a useful thing to know: changing an activation is a cheap experiment that adds nothing to the size of your model.

## Cleanup

This lab does not create anything that must be torn down, and part of what it produces may be kept. There are no processes to stop and no services to remove.

**The `data/` directory** holds the downloaded MNIST files. On a local machine, keeping it means the next chapter's load is instant instead of downloading again. If you need to reclaim disk space, you may delete `data/MNIST/`, but the dataset will download again the next time you load it.

On Google Colab the runtime is temporary. When your Colab session ends, the `data/` directory and the loaded `train_data` is cleared. This is expected. In the next chapter you simply re-run the setup cells, which downloads the data again for the new session.

## Summary

In this lab you built the `DigitClassifier`, opened it with `print(model)` and `model.named_parameters()`, and confirmed that its structure and its 101,770 learnable values matched what the code described. You then watched a Linear layer change a tensor's feature count while leaving the batch alone, passed a real MNIST image through the model to see it produce ten class scores, and compared how ReLU and Leaky ReLU treat the same negative values. Finally you changed the model twice, adding a layer and swapping an activation, and read each change back from the model: the extra layer raised the parameter count, and the new activation left it untouched.

The single skill underneath all of this is the one that carries into the rest of the course: make one deliberate change to a model, inspect it, and understand what changed before moving on. You now have a working `DigitClassifier` defined in your notebook and the MNIST dataset downloaded locally, which is the exact starting point Chapter 3 uses to put the model into training.  
